### what this model will do 
we will use NLP to check a given transaction description like "swiggy order" or "monthly house rent" 
the model will then predict the category like "food" and "rent"

pipeline would be something like:
raw test -> clean text -> TF-IDF numbers -> model -> categpry

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib

In [14]:
## improting data
df = pd.read_excel("../data/sample_transactions.xlsx")
df.head()

ValueError: Excel file format cannot be determined, you must specify an engine manually.

In [ ]:
## changing data from string to date format
df['date'] = pd.to_datetime(df['date'])

print(df['date'].head)
print(df['date'].dtype)

<bound method NDFrame.head of 0     2024-04-03
1     2024-03-25
2     2024-01-20
3     2024-10-10
4     2024-11-09
         ...    
535   2024-01-15
536   2024-09-25
537   2024-09-30
538   2024-08-26
539   2024-12-10
Name: date, Length: 540, dtype: datetime64[us]>
datetime64[us]


### what we are doing
train a model that predicts the spending category from the text description of the transaction

eg. input "Swiggy order" -> output "food"

In [ ]:
## observing the data
print(df[['description', 'category']].head(10))
print("\nUnique categories are: ", df["category"].nunique())
print(df["category"].value_counts())

                 description       category
0  Electric appliance repair      Utilities
1       Netflix subscription  Entertainment
2           Vegetable market      Groceries
3                 Yoga class        Fitness
4               Udemy course      Education
5            Amazon purchase       Shopping
6              Diesel refill           Fuel
7            Myntra shopping       Shopping
8                 Water bill          Bills
9             DMart shopping      Groceries

Unique categories are:  15
category
Utilities        36
Entertainment    36
Groceries        36
Fitness          36
Education        36
Shopping         36
Fuel             36
Bills            36
Personal Care    36
Insurance        36
Travel           36
Investments      36
Rent             36
Food             36
Healthcare       36
Name: count, dtype: int64


In [15]:
## data cleaning
df['clean_description'] = df['description'].str.lower().str.strip()

print(df[['description','clean_description']].head(10))

                 description          clean_description
0  Electric appliance repair  electric appliance repair
1       Netflix subscription       netflix subscription
2           Vegetable market           vegetable market
3                 Yoga class                 yoga class
4               Udemy course               udemy course
5            Amazon purchase            amazon purchase
6              Diesel refill              diesel refill
7            Myntra shopping            myntra shopping
8                 Water bill                 water bill
9             DMart shopping             dmart shopping


In [16]:
## making x-input and y-output

X = df['clean_description']
y = df['category']

print("Feature shape: ", X.shape)
print("target shape: ",y.shape)

Feature shape:  (540,)
target shape:  (540,)


In [17]:
## train test split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

print("training sample length ", len(X_train))
print("testing sample length ", len(X_test))

training sample length  432
testing sample length  108


In [18]:
# Vectorization using TF-IDF

vectorizer =  TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("vocabulary size: ", len(vectorizer.vocabulary_)) ## this will tell the number of unique words
print("training matrix shape: ", X_train_tfidf.shape)

vocabulary size:  127
training matrix shape:  (432, 127)


In [19]:
## training the model

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

print("model trained")

model trained


In [20]:
## evaluating the performance of model

y_pred = model.predict(X_test_tfidf)

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("\n", classification_report(y_test, y_pred))

Accuracy:  1.0

                precision    recall  f1-score   support

        Bills       1.00      1.00      1.00         8
    Education       1.00      1.00      1.00         5
Entertainment       1.00      1.00      1.00         5
      Fitness       1.00      1.00      1.00         6
         Food       1.00      1.00      1.00         5
         Fuel       1.00      1.00      1.00         9
    Groceries       1.00      1.00      1.00         8
   Healthcare       1.00      1.00      1.00         4
    Insurance       1.00      1.00      1.00         8
  Investments       1.00      1.00      1.00        11
Personal Care       1.00      1.00      1.00         9
         Rent       1.00      1.00      1.00        10
     Shopping       1.00      1.00      1.00         6
       Travel       1.00      1.00      1.00         7
    Utilities       1.00      1.00      1.00         7

     accuracy                           1.00       108
    macro avg       1.00      1.00      1.00  